# Rolling-Horizon EV Charging Control

This notebook tests charging decisions made step by step.

## Purpose

Run charging control in rolling steps. Each step looks ahead, applies the first part of the schedule, updates the remaining energy, and solves again.

In [1]:
from pathlib import Path
import json
import math
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from either the root or Notebooks directory."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Data").exists() and (candidate / "Notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing Data/ and Notebooks/.")


ROOT = find_project_root()
DATA_DIR = ROOT / "Data"
RESULTS_DIR = DATA_DIR / "Results"
PROCESSED_DIR = DATA_DIR / "Processed"
FIGURES_DIR = ROOT / "Figures"
MODELS_DIR = ROOT / "Models" / "Forecasting"

for directory in [RESULTS_DIR, PROCESSED_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

INTERVAL_MINUTES = 5
INTERVAL_HOURS = INTERVAL_MINUTES / 60
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print(f"Project root: {ROOT}")

Project root: /Users/mac/Send-Ev-Project/SEND-EV-Project


In [2]:
def save_table(df: pd.DataFrame, path: Path) -> Path:
    """Save Parquet where available, otherwise save CSV with the same stem."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        try:
            df.to_parquet(path, index=False)
            print(f"Saved: {path.relative_to(ROOT)}")
            return path
        except (ImportError, ModuleNotFoundError) as exc:
            csv_path = path.with_suffix(".csv")
            df.to_csv(csv_path, index=False)
            print(f"Parquet engine unavailable ({exc.__class__.__name__}); saved CSV: {csv_path.relative_to(ROOT)}")
            return csv_path
    df.to_csv(path, index=False)
    print(f"Saved: {path.relative_to(ROOT)}")
    return path


def read_table(path: Path, parse_dates: list[str] | None = None) -> pd.DataFrame:
    """Read a Parquet/CSV output regardless of which format was written."""
    candidates = [path]
    if path.suffix.lower() == ".parquet":
        candidates.append(path.with_suffix(".csv"))
    elif path.suffix.lower() == ".csv":
        candidates.append(path.with_suffix(".parquet"))

    parquet_error: Exception | None = None

    for candidate in candidates:
        if candidate.exists():
            if candidate.suffix.lower() == ".parquet":
                try:
                    return pd.read_parquet(candidate)
                except (ImportError, ModuleNotFoundError) as exc:
                    parquet_error = exc
                    continue
            return pd.read_csv(candidate, parse_dates=parse_dates)

    if parquet_error is not None:
        raise ImportError(
            "A Parquet file exists, but pandas cannot read it and no CSV fallback was found. "
            "Install pyarrow or fastparquet, or rerun the previous notebook to create the CSV fallback."
        ) from parquet_error

    raise FileNotFoundError(f"None of these files exists: {candidates}")

In [3]:
try:
    import cvxpy as cp
except ImportError as exc:
    raise ImportError("Install CVXPY with: poetry add cvxpy") from exc

scenarios = read_table(DATA_DIR / "ACN" / "processed" / "keele_ev_scenarios.parquet")
send = read_table(PROCESSED_DIR / "send_solcast_surplus_5min.parquet", parse_dates=["DateTime"])
for column in ["arrival_time", "departure_time", "keele_date"]:
    scenarios[column] = pd.to_datetime(scenarios[column], errors="coerce")
send["DateTime"] = pd.to_datetime(send["DateTime"], errors="coerce")

# Add forecast columns if they are saved separately.
for prediction_name in ["surplus_predictions.csv", "surplus_test_features.csv", "forecast_predictions.csv"]:
    prediction_path = RESULTS_DIR / prediction_name
    if not prediction_path.exists():
        continue
    predictions = pd.read_csv(prediction_path, parse_dates=["DateTime"])
    prediction_columns = [c for c in ["surplus_pred_xgb", "surplus_pred_lgbm"] if c in predictions.columns]
    if prediction_columns:
        send = send.merge(
            predictions[["DateTime", *prediction_columns]].drop_duplicates("DateTime"),
            on="DateTime", how="left", suffixes=("", "_new")
        )
        for column in prediction_columns:
            new_column = f"{column}_new"
            if new_column in send.columns:
                send[column] = send[column].combine_first(send[new_column]) if column in send.columns else send[new_column]
                send = send.drop(columns=[new_column])
        print(f"Merged forecast columns from {prediction_name}: {prediction_columns}")

In [4]:
def scenario_time_grid(group: pd.DataFrame, send: pd.DataFrame) -> pd.DatetimeIndex:
    start = pd.Timestamp(group["keele_date"].iloc[0]).normalize()
    latest_departure = pd.to_datetime(group["departure_time"]).max()
    # Use at most 48 hours of SEND data.
    end = min(latest_departure.ceil(f"{INTERVAL_MINUTES}min"), start + pd.Timedelta(hours=48))
    available_end = send["DateTime"].max() + pd.Timedelta(minutes=INTERVAL_MINUTES)
    end = min(end, available_end)
    return pd.date_range(start, end, freq=f"{INTERVAL_MINUTES}min", inclusive="left")


def aligned_send_series(send: pd.DataFrame, grid: pd.DatetimeIndex, column: str) -> np.ndarray:
    series = send.set_index("DateTime")[column].reindex(grid)
    # Fill short gaps; set any remaining gaps to zero.
    return series.interpolate(limit=3).fillna(0).to_numpy(dtype=float)


def schedule_metrics(
    scenario: pd.DataFrame,
    vehicle_schedule: pd.DataFrame,
    send: pd.DataFrame,
    strategy: str,
    forecast_source: str = "none",
) -> dict:
    grid = scenario_time_grid(scenario, send)
    surplus = aligned_send_series(send, grid, "surplus_kw")
    consumption = aligned_send_series(send, grid, "consumption_kw")

    if vehicle_schedule.empty:
        aggregate = pd.Series(0.0, index=grid)
        delivered_by_vehicle = pd.Series(dtype=float)
    else:
        schedule = vehicle_schedule.copy()
        schedule["DateTime"] = pd.to_datetime(schedule["DateTime"])
        aggregate = schedule.groupby("DateTime")["charging_kw"].sum().reindex(grid, fill_value=0.0)
        delivered_by_vehicle = (
            schedule.groupby("vehicle_id")["charging_kw"].sum() * INTERVAL_HOURS
        )

    required = scenario.set_index("vehicle_id")["energy_required_kwh"]
    delivered = delivered_by_vehicle.reindex(required.index, fill_value=0.0)
    service_rate = float((delivered >= required * 0.99).mean())
    unserved = float((required - delivered).clip(lower=0).sum())

    renewable_absorbed = np.minimum(aggregate.to_numpy(), surplus).sum() * INTERVAL_HOURS
    residual_surplus = np.maximum(surplus - aggregate.to_numpy(), 0).sum() * INTERVAL_HOURS
    ev_grid_energy = np.maximum(aggregate.to_numpy() - surplus, 0).sum() * INTERVAL_HOURS

    return {
        "scenario_id": scenario["scenario_id"].iloc[0],
        "keele_date": scenario["keele_date"].iloc[0],
        "scenario_site": scenario["scenario_site"].iloc[0],
        "fleet_size": int(scenario["fleet_size"].iloc[0]),
        "participation_rate": float(scenario["participation_rate"].iloc[0]),
        "participant_count": int(scenario["participant_count"].iloc[0]),
        "site_limit_kw": float(scenario["site_limit_kw"].iloc[0]),
        "strategy": strategy,
        "forecast_source": forecast_source,
        "required_energy_kwh": float(required.sum()),
        "delivered_energy_kwh": float(delivered.sum()),
        "unserved_energy_kwh": unserved,
        "service_rate": service_rate,
        "renewable_absorbed_kwh": float(renewable_absorbed),
        "residual_surplus_kwh": float(residual_surplus),
        "ev_grid_energy_kwh": float(ev_grid_energy),
        "peak_ev_kw": float(aggregate.max()),
        "peak_total_load_kw": float((aggregate.to_numpy() + consumption).max()),
    }

## 1. Rolling-control configuration

In [5]:
HORIZON_HOURS = 6
HORIZON_STEPS = int(HORIZON_HOURS / INTERVAL_HOURS)
REOPTIMISE_EVERY_STEPS = 3  # every 15 minutes
MAX_SCENARIOS = 6
INCLUDE_KNOWN_FUTURE_ARRIVALS = True

FORECAST_COLUMN_CANDIDATES = {
    "xgb": "surplus_pred_xgb",
    "lgbm": "surplus_pred_lgbm",
    "actual": "surplus_kw",
}
available_sources = {
    name: column for name, column in FORECAST_COLUMN_CANDIDATES.items() if column in send.columns
}

# Use forecasts when available; keep actual data as a benchmark.
ROLLING_SOURCES = [s for s in ["xgb", "lgbm", "actual"] if s in available_sources]
print("Rolling sources:", ROLLING_SOURCES)

Rolling sources: ['actual']


## 2. Rolling-horizon solver

In [6]:
def solve_control_window(
    vehicles: pd.DataFrame,
    remaining_kwh: np.ndarray,
    grid_window: pd.DatetimeIndex,
    signal: np.ndarray,
    site_limit_kw: float,
) -> np.ndarray:
    n_vehicles, n_steps = len(vehicles), len(grid_window)
    if n_vehicles == 0 or n_steps == 0:
        return np.zeros((n_vehicles, n_steps))

    arrival = vehicles["arrival_time"].to_numpy(dtype="datetime64[ns]")
    departure = vehicles["departure_time"].to_numpy(dtype="datetime64[ns]")
    grid_values = grid_window.to_numpy(dtype="datetime64[ns]")
    availability = (
        (grid_values[None, :] >= arrival[:, None])
        & (grid_values[None, :] < departure[:, None])
    ).astype(float)

    max_power = vehicles["max_power_kw"].to_numpy(dtype=float)
    efficiency = vehicles["efficiency"].to_numpy(dtype=float)
    window_end = grid_window[-1] + pd.Timedelta(minutes=INTERVAL_MINUTES)

    # Charge enough now so the vehicle can still finish later.
    future_hours = np.maximum(
        (vehicles["departure_time"] - window_end).dt.total_seconds().to_numpy() / 3600,
        0,
    )
    future_capacity = future_hours * max_power * efficiency
    mandatory_window_energy = np.maximum(remaining_kwh - future_capacity, 0)

    charging = cp.Variable((n_vehicles, n_steps), nonneg=True)
    shortfall = cp.Variable(n_vehicles, nonneg=True)
    aggregate = cp.sum(charging, axis=0)
    delivered_window = cp.multiply(efficiency, cp.sum(charging, axis=1) * INTERVAL_HOURS)

    constraints = [
        charging <= max_power[:, None] * availability,
        aggregate <= site_limit_kw,
        delivered_window + shortfall >= mandatory_window_energy,
    ]

    smoothness = cp.sum_squares(aggregate[1:] - aggregate[:-1]) if n_steps > 1 else 0
    objective = cp.Minimize(
        10_000 * cp.sum(shortfall)
        + cp.sum(cp.pos(signal - aggregate)) * INTERVAL_HOURS
        + 5 * cp.sum(cp.pos(aggregate - signal)) * INTERVAL_HOURS
        + 0.001 * smoothness
    )
    problem = cp.Problem(objective, constraints)

    installed = set(cp.installed_solvers())
    for solver in ["CLARABEL", "OSQP", "SCS"]:
        if solver not in installed:
            continue
        try:
            problem.solve(solver=solver, verbose=False)
            if problem.status in {cp.OPTIMAL, cp.OPTIMAL_INACCURATE}:
                return np.maximum(np.asarray(charging.value), 0)
        except Exception:
            continue
    raise RuntimeError(f"Rolling optimisation failed: {problem.status}")


def run_rolling_scenario(
    scenario: pd.DataFrame,
    send: pd.DataFrame,
    source: str,
    forecast_column: str,
) -> tuple[pd.DataFrame, dict]:
    group = scenario.copy().sort_values("vehicle_id").reset_index(drop=True)
    full_grid = scenario_time_grid(group, send)
    signal_full = aligned_send_series(send, full_grid, forecast_column)
    remaining = group["energy_required_kwh"].to_numpy(dtype=float).copy()
    records = []

    step = 0
    while step < len(full_grid):
        current_time = full_grid[step]
        horizon_end = min(step + HORIZON_STEPS, len(full_grid))
        window = full_grid[step:horizon_end]
        signal = signal_full[step:horizon_end]

        if INCLUDE_KNOWN_FUTURE_ARRIVALS:
            active_mask = group["departure_time"].gt(current_time) & group["arrival_time"].lt(
                window[-1] + pd.Timedelta(minutes=INTERVAL_MINUTES)
            )
        else:
            active_mask = group["departure_time"].gt(current_time) & group["arrival_time"].le(current_time)
        active_mask &= remaining > 1e-8

        active_indices = np.flatnonzero(active_mask.to_numpy())
        action_steps = min(REOPTIMISE_EVERY_STEPS, len(window))
        if len(active_indices) == 0:
            step += action_steps
            continue

        active_vehicles = group.iloc[active_indices].reset_index(drop=True)
        planned = solve_control_window(
            active_vehicles,
            remaining[active_indices],
            window,
            signal,
            float(group["site_limit_kw"].iloc[0]),
        )

        for local_vehicle_idx, global_vehicle_idx in enumerate(active_indices):
            vehicle = group.iloc[global_vehicle_idx]
            for offset in range(action_steps):
                charging_kw = planned[local_vehicle_idx, offset]
                if charging_kw <= 1e-5:
                    continue
                timestamp = window[offset]
                delivered = charging_kw * INTERVAL_HOURS * float(vehicle["efficiency"])
                delivered = min(delivered, remaining[global_vehicle_idx])
                charging_kw = delivered / (INTERVAL_HOURS * float(vehicle["efficiency"]))
                remaining[global_vehicle_idx] -= delivered
                records.append({
                    "scenario_id": vehicle["scenario_id"],
                    "vehicle_id": vehicle["vehicle_id"],
                    "DateTime": timestamp,
                    "charging_kw": charging_kw,
                    "strategy": f"rolling_{source}",
                    "forecast_source": source,
                })
        step += action_steps

    schedule = pd.DataFrame(records)
    metrics = schedule_metrics(group, schedule, send, f"rolling_{source}", source)
    metrics["remaining_energy_state_kwh"] = float(np.maximum(remaining, 0).sum())
    metrics["horizon_hours"] = HORIZON_HOURS
    metrics["reoptimise_minutes"] = REOPTIMISE_EVERY_STEPS * INTERVAL_MINUTES
    return schedule, metrics

## 3. Run rolling experiments

In [7]:
scenario_ids = scenarios["scenario_id"].drop_duplicates().tolist()[:MAX_SCENARIOS]
schedule_frames = []
metric_rows = []

for count, scenario_id in enumerate(scenario_ids, start=1):
    scenario = scenarios[scenarios["scenario_id"].eq(scenario_id)].copy()
    for source in ROLLING_SOURCES:
        schedule, metrics = run_rolling_scenario(
            scenario, send, source, available_sources[source]
        )
        schedule_frames.append(schedule)
        metric_rows.append(metrics)
    print(f"Completed rolling scenario {count}/{len(scenario_ids)}")

rolling_schedule = pd.concat(schedule_frames, ignore_index=True) if schedule_frames else pd.DataFrame()
rolling_metrics = pd.DataFrame(metric_rows)
display(rolling_metrics)

Completed rolling scenario 1/6
Completed rolling scenario 2/6
Completed rolling scenario 3/6
Completed rolling scenario 4/6


/Users/mac/miniforge3/envs/send-ev/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Completed rolling scenario 5/6


/Users/mac/miniforge3/envs/send-ev/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Completed rolling scenario 6/6


,scenario_id,keele_date,scenario_site,fleet_size,participation_rate,participant_count,site_limit_kw,strategy,forecast_source,required_energy_kwh,delivered_energy_kwh,unserved_energy_kwh,service_rate,renewable_absorbed_kwh,residual_surplus_kwh,ev_grid_energy_kwh,peak_ev_kw,peak_total_load_kw,remaining_energy_state_kwh,horizon_hours,reoptimise_minutes
0,20230707_pooled_F10_P025_S42,2023-07-07,pooled,10,0.25,2,50.0,rolling_actual,actual,14.725,16.361111,0.0,1.0,16.361111,12260.382705,0.00,13.200000,2062.154000,0.000,6,15
1,20230707_pooled_F10_P025_S43,2023-07-07,pooled,10,0.25,2,50.0,rolling_actual,actual,16.763,18.625556,0.0,1.0,18.625556,12695.190200,0.00,13.199999,2068.754000,0.000,6,15
2,20230707_pooled_F10_P050_S42,2023-07-07,pooled,10,0.50,5,50.0,rolling_actual,actual,64.482,71.426667,0.0,1.0,34.576667,12679.239089,36.85,13.200000,2062.154000,0.198,6,15
3,20230707_pooled_F10_P050_S43,2023-07-07,pooled,10,0.50,5,50.0,rolling_actual,actual,39.417,43.796667,0.0,1.0,33.016667,12680.799089,10.78,13.199998,2068.753999,0.000,6,15
4,20230707_pooled_F10_P075_S42,2023-07-07,pooled,10,0.75,8,50.0,rolling_actual,actual,96.707,107.232222,0.0,1.0,70.382222,12643.433534,36.85,19.800000,2075.353999,0.198,6,15
5,20230707_pooled_F10_P075_S43,2023-07-07,pooled,10,0.75,8,50.0,rolling_actual,actual,55.874,62.082222,0.0,1.0,51.302222,12662.513534,10.78,19.799999,2068.753998,0.000,6,15


## 4. Save outputs

In [8]:
save_table(rolling_schedule, RESULTS_DIR / "rolling_vehicle_schedule.parquet")
save_table(rolling_metrics, RESULTS_DIR / "rolling_horizon_metrics.csv")

Saved: Data/Results/rolling_vehicle_schedule.parquet
Saved: Data/Results/rolling_horizon_metrics.csv


PosixPath('/Users/mac/Send-Ev-Project/SEND-EV-Project/Data/Results/rolling_horizon_metrics.csv')